<a href="https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The playbook scores April content items using a Random Forest trained on February features and the defined March position-adjusted CTR proxy. The model risk score estimates how likely an item is to fall below the typical CTR for its comparable search-position bucket; the action score also accounts for observed April impressions.

- **REVIEW_CTR_OPPORTUNITY:** High predicted CTR-risk, average April position 4–20, and meaningful search visibility. A human reviews title, snippet, search intent, and page context.
- **INVESTIGATE_HIGH_RISK:** High predicted CTR-risk outside the 4–20 position range. A human diagnoses whether the issue is visibility, intent, tracking, or another factor before editing.
- **MONITOR:** Lower-priority candidates. No automatic change is made.

Reason codes describe why an item enters a review queue. They do not assert that a proposed change will increase clicks.

In [9]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{hf_token}')"
)

FEB = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-02/*.parquet"
)

MARCH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

APRIL = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/*.parquet"
)

# February features and the later March position-adjusted CTR proxy label.
training_df = con.execute(f"""
    WITH february_features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS feb_impressions,
            AVG(NULLIF(gsc_avg_position, 0)) AS feb_avg_position,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN scroll_events END)
                AS feb_scroll_events,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_social END)
                AS feb_sessions_social,
            COUNT(*) AS feb_observed_days,
            MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
                AS has_ga4_data
        FROM read_parquet('{FEB}')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    march_outcomes AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions,
            SUM(gsc_clicks)::DOUBLE
                / NULLIF(SUM(gsc_impressions), 0) AS march_ctr,
            AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position
        FROM read_parquet('{MARCH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    march_bucketed AS (
        SELECT *,
            CASE
                WHEN march_avg_position <= 3 THEN '1-3'
                WHEN march_avg_position <= 10 THEN '4-10'
                WHEN march_avg_position <= 20 THEN '11-20'
                WHEN march_avg_position <= 50 THEN '21-50'
                ELSE '51+'
            END AS march_position_bucket
        FROM march_outcomes
        WHERE march_impressions >= 100
          AND march_avg_position IS NOT NULL
    ),
    labelled_march AS (
        SELECT *,
            MEDIAN(march_ctr) OVER (
                PARTITION BY march_position_bucket
            ) AS march_bucket_median_ctr
        FROM march_bucketed
    )
    SELECT
        f.*,
        CASE
            WHEN f.feb_avg_position <= 3 THEN '1-3'
            WHEN f.feb_avg_position <= 10 THEN '4-10'
            WHEN f.feb_avg_position <= 20 THEN '11-20'
            WHEN f.feb_avg_position <= 50 THEN '21-50'
            ELSE 'missing_or_51+'
        END AS feb_position_bucket,
        CASE
            WHEN m.march_ctr < m.march_bucket_median_ctr THEN 1
            ELSE 0
        END AS target
    FROM february_features AS f
    INNER JOIN labelled_march AS m
        USING (client_hash_id, content_hash_id)
    WHERE f.feb_impressions >= 100
      AND f.feb_avg_position IS NOT NULL
""").df()

feature_columns = [
    "feb_impressions",
    "feb_avg_position",
    "feb_scroll_events",
    "feb_sessions_social",
    "feb_observed_days",
    "has_ga4_data",
    "feb_position_bucket",
]

numeric_features = [
    "feb_impressions",
    "feb_avg_position",
    "feb_scroll_events",
    "feb_sessions_social",
    "feb_observed_days",
    "has_ga4_data",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    (
                        "impute",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True,
                        ),
                    ),
                ]
            ),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("encode", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            ["feb_position_bucket"],
        ),
    ]
)

playbook_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "random_forest",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=20,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

playbook_model.fit(
    training_df[feature_columns],
    training_df["target"],
)

# April scoring frame: renamed columns fit the February-feature pipeline.
april_df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        AVG(NULLIF(gsc_avg_position, 0)) AS april_avg_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN scroll_events END)
            AS april_scroll_events,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_social END)
            AS april_sessions_social,
        COUNT(*) AS april_observed_days,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
            AS has_ga4_data
    FROM read_parquet('{APRIL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
       AND AVG(NULLIF(gsc_avg_position, 0)) IS NOT NULL
""").df()

scoring_features = pd.DataFrame(
    {
        "feb_impressions": april_df["april_impressions"],
        "feb_avg_position": april_df["april_avg_position"],
        "feb_scroll_events": april_df["april_scroll_events"],
        "feb_sessions_social": april_df["april_sessions_social"],
        "feb_observed_days": april_df["april_observed_days"],
        "has_ga4_data": april_df["has_ga4_data"],
        "feb_position_bucket": pd.cut(
            april_df["april_avg_position"],
            bins=[0, 3, 10, 20, 50, np.inf],
            labels=["1-3", "4-10", "11-20", "21-50", "missing_or_51+"],
            include_lowest=True,
        ).astype(str),
    }
)

april_df["model_risk_score"] = playbook_model.predict_proba(
    scoring_features[feature_columns]
)[:, 1]

# Top 10% model-risk threshold within the April scoring population.
risk_threshold = april_df["model_risk_score"].quantile(0.90)

high_risk = april_df["model_risk_score"] >= risk_threshold
striking_distance = april_df["april_avg_position"].between(4, 20)

april_df["reason_code"] = np.select(
    [
        high_risk & striking_distance,
        high_risk,
    ],
    [
        "HIGH_RISK_STRIKING_DISTANCE",
        "HIGH_RISK_OUTSIDE_STRIKING_DISTANCE",
    ],
    default="LOWER_PRIORITY_RISK",
)

april_df["action_label"] = np.select(
    [
        high_risk & striking_distance,
        high_risk,
    ],
    [
        "REVIEW_CTR_OPPORTUNITY",
        "INVESTIGATE_HIGH_RISK",
    ],
    default="MONITOR",
)

# Only high-risk items receive an actionable ranking score.
april_df["action_score"] = np.where(
    high_risk,
    april_df["model_risk_score"] * np.log1p(april_df["april_impressions"]),
    0.0,
)

april_df["action_priority"] = april_df["action_label"].map(
    {
        "REVIEW_CTR_OPPORTUNITY": 2,
        "INVESTIGATE_HIGH_RISK": 1,
        "MONITOR": 0,
    }
)

ranked_actions = april_df.sort_values(
    ["action_priority", "action_score", "model_risk_score"],
    ascending=[False, False, False],
).reset_index(drop=True)

print("=== April action-playbook queue ===")
print(f"Scored April content items: {len(ranked_actions):,}")
print(f"High-risk threshold (top 10%): {risk_threshold:.3f}")

display(
    ranked_actions["action_label"]
    .value_counts()
    .rename_axis("action_label")
    .reset_index(name="items")
)

display(
    ranked_actions[
        [
            "content_hash_id",
            "client_hash_id",
            "april_impressions",
            "april_avg_position",
            "model_risk_score",
            "action_score",
            "reason_code",
            "action_label",
        ]
    ].head(20)
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== April action-playbook queue ===
Scored April content items: 107,144
High-risk threshold (top 10%): 0.646


,action_label,items
0,MONITOR,96429
1,REVIEW_CTR_OPPORTUNITY,9253
2,INVESTIGATE_HIGH_RISK,1462


,content_hash_id,client_hash_id,april_impressions,april_avg_position,model_risk_score,action_score,reason_code,action_label
0,content_36fc1ee501ec072d,client_62f4a7e64f5e0096,55225.0,8.467967,0.714019,7.796508,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY
1,content_9111c7d2691be9ad,client_62f4a7e64f5e0096,128006.0,7.242113,0.658317,7.741699,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY
2,content_b99ea6861864dea5,client_62f4a7e64f5e0096,141360.0,6.901627,0.650875,7.718779,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY
3,content_23a42776a7009b65,client_73cda7b4e4f265ea,91524.0,9.400853,0.675464,7.716750,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY
4,content_851a5988430e67b8,client_62f4a7e64f5e0096,42679.0,8.475237,0.715119,7.624233,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY
5,content_77b413ea9cd9f030,client_62f4a7e64f5e0096,46317.0,8.113097,0.707793,7.604025,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY
6,content_ca0aa83ba028d4a2,client_08a6a72ff48e62c0,40183.0,8.457793,0.716206,7.592664,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY
7,content_046fc480045b88f5,client_a80fca3f171ed1de,46327.0,7.782806,0.705336,7.577774,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY
8,content_7ff032d3024d35e4,client_62f4a7e64f5e0096,38712.0,8.649743,0.713210,7.534298,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY
9,content_132dcd40faff1dc2,client_62f4a7e64f5e0096,50258.0,9.027467,0.694503,7.517959,HIGH_RISK_STRIKING_DISTANCE,REVIEW_CTR_OPPORTUNITY


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

**Intended users:** SEO managers, content strategists, and editors who decide which content items deserve manual review first.

**Intended use:** use the ranked April queue to select a small weekly review batch. For `REVIEW_CTR_OPPORTUNITY` items, a person checks search intent, title and snippet quality, page context, and tracking before proposing any change.

**Operating limits:** this playbook applies only to the observed April scoring population with Search Console availability, at least 100 impressions, and a valid average position. It is not a recommendation for new or low-data items, and it does not automate edits, URLs, publishing decisions, or claims of click improvement. The model was trained on one February-to-March historical relationship and requires future monitoring before any production use.

In [10]:
# Verify the actual scoring population and operating limits

limit_summary = pd.DataFrame(
    {
        "check": [
            "Scored April items",
            "Minimum April impressions in scored population",
            "Items missing April average position",
            "Items without eligible April GA4 data",
            "Review actions requiring human review",
        ],
        "observed_value": [
            len(ranked_actions),
            int(ranked_actions["april_impressions"].min()),
            int(ranked_actions["april_avg_position"].isna().sum()),
            int((ranked_actions["has_ga4_data"] == 0).sum()),
            int(
                ranked_actions["action_label"]
                .isin(
                    [
                        "REVIEW_CTR_OPPORTUNITY",
                        "INVESTIGATE_HIGH_RISK",
                    ]
                )
                .sum()
            ),
        ],
    }
)

print("=== Intended-use and operating-limit checks ===")
display(limit_summary)

print(
    "\nAll scored items meet the April eligibility rules: "
    "Search Console availability, at least 100 impressions, and a valid "
    "average position. Missing GA4 engagement data is handled by the "
    "model's imputation pipeline, not treated as a zero-value signal."
)

=== Intended-use and operating-limit checks ===


,check,observed_value
0,Scored April items,107144
1,Minimum April impressions in scored population,100
2,Items missing April average position,0
3,Items without eligible April GA4 data,40984
4,Review actions requiring human review,10715



All scored items meet the April eligibility rules: Search Console availability, at least 100 impressions, and a valid average position. Missing GA4 engagement data is handled by the model's imputation pipeline, not treated as a zero-value signal.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review and the no-go list

Before acting on a review item, an editor checks the page’s search intent, title and snippet, page content, user experience, tracking quality, and any legal or brand requirements. The score only prioritizes attention; it does not diagnose the cause of a CTR gap.

**Never automate:** title or metadata changes, page-content rewrites, URL changes, redirects, canonical tags, publishing or deletion decisions, legal/compliance content, or brand-sensitive content. Every action remains a human decision with a documented review.

In [11]:
# Apply human-review guardrails to every playbook recommendation

actionable_labels = [
    "REVIEW_CTR_OPPORTUNITY",
    "INVESTIGATE_HIGH_RISK",
]

ranked_actions["requires_human_review"] = ranked_actions[
    "action_label"
].isin(actionable_labels)

ranked_actions["automation_status"] = "NO_AUTOMATIC_CHANGES"

guardrail_summary = (
    ranked_actions.groupby(
        ["action_label", "requires_human_review", "automation_status"],
        dropna=False,
    )
    .size()
    .reset_index(name="items")
)

no_go_list = pd.DataFrame(
    {
        "never_automate": [
            "Title or metadata changes",
            "Page-content rewrites",
            "URLs, redirects, canonicals, or publishing status",
            "Legal, compliance, or brand-sensitive content",
        ],
        "required_handling": [
            "Human editorial review before any proposed change",
            "Human editorial review before any proposed change",
            "Human technical review before any proposed change",
            "Specialist or owner approval before any proposed change",
        ],
    }
)

print("=== Human-review guardrails ===")
display(guardrail_summary)

print("=== No-go list ===")
display(no_go_list)

assert (ranked_actions["automation_status"] == "NO_AUTOMATIC_CHANGES").all()

print(
    "Guardrail passed: the playbook produces prioritization labels only. "
    "It performs no automated edit, URL, publishing, or deletion action."
)

=== Human-review guardrails ===


,action_label,requires_human_review,automation_status,items
0,INVESTIGATE_HIGH_RISK,True,NO_AUTOMATIC_CHANGES,1462
1,MONITOR,False,NO_AUTOMATIC_CHANGES,96429
2,REVIEW_CTR_OPPORTUNITY,True,NO_AUTOMATIC_CHANGES,9253


=== No-go list ===


,never_automate,required_handling
0,Title or metadata changes,Human editorial review before any proposed change
1,Page-content rewrites,Human editorial review before any proposed change
2,"URLs, redirects, canonicals, or publishing status",Human technical review before any proposed change
3,"Legal, compliance, or brand-sensitive content",Specialist or owner approval before any propos...


Guardrail passed: the playbook produces prioritization labels only. It performs no automated edit, URL, publishing, or deletion action.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

The April action queue is a baseline for later monitoring, not evidence of future impact. When a later complete observation window becomes available, I will compare its data coverage, feature distributions, score distribution, and action-volume rate with this April baseline.

I will investigate or retrain when there is material missing-data growth, a large shift in model-score or action-label distribution, a sustained drop in held-out ranking quality, or a meaningful change in the data-generation process. Retraining requires repeating the grouped-client validation and leakage audit; it is not an automatic monthly refresh.

In [12]:
# April monitoring baseline: records what future scoring runs should be compared against

review_count = int(
    (ranked_actions["action_label"] == "REVIEW_CTR_OPPORTUNITY").sum()
)

investigate_count = int(
    (ranked_actions["action_label"] == "INVESTIGATE_HIGH_RISK").sum()
)

missing_ga4_rate = float(
    (ranked_actions["has_ga4_data"] == 0).mean()
)

monitoring_baseline = pd.DataFrame(
    {
        "baseline_measure": [
            "Scored April items",
            "High-risk model-score threshold",
            "Review action rate",
            "Investigate action rate",
            "Missing eligible GA4-data rate",
            "Median April impressions",
            "Median model risk score",
        ],
        "observed_value": [
            len(ranked_actions),
            risk_threshold,
            review_count / len(ranked_actions),
            investigate_count / len(ranked_actions),
            missing_ga4_rate,
            ranked_actions["april_impressions"].median(),
            ranked_actions["model_risk_score"].median(),
        ],
    }
)

retrain_triggers = pd.DataFrame(
    {
        "monitoring_area": [
            "Data coverage",
            "Feature or score distribution",
            "Action-volume distribution",
            "Held-out validation quality",
            "Data-generation process",
        ],
        "investigate_or_retrain_when": [
            "Missingness or eligibility coverage materially changes from the April baseline.",
            "Feature or model-score distributions shift materially from the April baseline.",
            "Review or investigate action rates change materially without an understood data reason.",
            "A fresh grouped-client validation is materially worse than the current held-out benchmark.",
            "Definitions, logging, availability flags, or warehouse schema change.",
        ],
    }
)

print("=== April monitoring baseline ===")
display(monitoring_baseline.round(4))

print("=== Investigate / retrain triggers ===")
display(retrain_triggers)

print(
    "No later complete scoring window is evaluated in this notebook. "
    "These April values are baseline measurements for a future comparison."
)

=== April monitoring baseline ===


,baseline_measure,observed_value
0,Scored April items,107144.0000
1,High-risk model-score threshold,0.6457
2,Review action rate,0.0864
3,Investigate action rate,0.0136
4,Missing eligible GA4-data rate,0.3825
5,Median April impressions,691.0000
6,Median model risk score,0.4311


=== Investigate / retrain triggers ===


,monitoring_area,investigate_or_retrain_when
0,Data coverage,Missingness or eligibility coverage materially...
1,Feature or score distribution,Feature or model-score distributions shift mat...
2,Action-volume distribution,Review or investigate action rates change mate...
3,Held-out validation quality,A fresh grouped-client validation is materiall...
4,Data-generation process,"Definitions, logging, availability flags, or w..."


No later complete scoring window is evaluated in this notebook. These April values are baseline measurements for a future comparison.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Public-safe paper export

The paper needs reproducible aggregate evidence, not a row-level recommendation queue. I export a small JSON summary containing only aggregate action counts, model threshold, and monitoring baselines.

No content identifiers, client identifiers, URLs, raw warehouse data, or row-level scored queue are exported or committed.

In [13]:
from pathlib import Path
import json

# Export only a public-safe aggregate summary.
# Do not export or commit the row-level ranked_actions queue.

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

summary_path = output_dir / "action_playbook_summary.json"

action_counts = (
    ranked_actions["action_label"]
    .value_counts()
    .sort_index()
    .to_dict()
)

paper_summary = {
    "data_window": "April 2026 aggregated scoring snapshot",
    "scored_items": int(len(ranked_actions)),
    "high_risk_threshold": round(float(risk_threshold), 4),
    "action_counts": {
        key: int(value)
        for key, value in action_counts.items()
    },
    "review_action_rate": round(
        float(
            (
                ranked_actions["action_label"]
                == "REVIEW_CTR_OPPORTUNITY"
            ).mean()
        ),
        4,
    ),
    "investigate_action_rate": round(
        float(
            (
                ranked_actions["action_label"]
                == "INVESTIGATE_HIGH_RISK"
            ).mean()
        ),
        4,
    ),
    "median_april_impressions": round(
        float(ranked_actions["april_impressions"].median()),
        2,
    ),
    "median_model_risk_score": round(
        float(ranked_actions["model_risk_score"].median()),
        4,
    ),
    "guardrail": (
        "Recommendations are prioritization labels only; "
        "no automated site changes are performed."
    ),
    "limitations": (
        "This is an observational decision-support summary, not a causal "
        "claim that an edit will increase clicks."
    ),
}

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(paper_summary, file, indent=2)

print("=== Public-safe paper export ===")
print(f"Saved summary: {summary_path}")
print(json.dumps(paper_summary, indent=2))

assert "content_hash_id" not in paper_summary
assert "client_hash_id" not in paper_summary

print(
    "\nExport check passed: only aggregate, public-safe metrics were written. "
    "No row-level queue was exported."
)

=== Public-safe paper export ===
Saved summary: work/outputs/action_playbook_summary.json
{
  "data_window": "April 2026 aggregated scoring snapshot",
  "scored_items": 107144,
  "high_risk_threshold": 0.6457,
  "action_counts": {
    "INVESTIGATE_HIGH_RISK": 1462,
    "MONITOR": 96429,
    "REVIEW_CTR_OPPORTUNITY": 9253
  },
  "review_action_rate": 0.0864,
  "investigate_action_rate": 0.0136,
  "median_april_impressions": 691.0,
  "median_model_risk_score": 0.4311,
  "guardrail": "Recommendations are prioritization labels only; no automated site changes are performed.",
  "limitations": "This is an observational decision-support summary, not a causal claim that an edit will increase clicks."
}

Export check passed: only aggregate, public-safe metrics were written. No row-level queue was exported.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.